# Трёхуровневое преобразование Хаара и сжатие изображения

**Задача:** выполнить трёхуровневое двумерное преобразование Хаара для стандартного изображения размером 512 × 512 пикселей, выполнить пороговое сжатие коэффициентов и восстановить изображение.

В работе используется изображение `camera` из библиотеки `scikit-image`. Для преобразования применяется библиотека `PyWavelets`.

## 1. Установка и импорт библиотек

Если библиотеки уже установлены, ячейку установки можно пропустить.

In [ ]:
%pip install -q numpy matplotlib scikit-image PyWavelets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

from skimage import data
from skimage.metrics import peak_signal_noise_ratio, mean_squared_error

%matplotlib inline

## 2. Загрузка исходного изображения

Изображение `camera` — стандартный тестовый снимок из `scikit-image`. Проверим его размер и тип данных.

In [ ]:
image = data.camera()

print("Размер изображения:", image.shape)
print("Тип данных:", image.dtype)
print("Минимальная яркость:", image.min())
print("Максимальная яркость:", image.max())

assert image.shape == (512, 512), "Ожидалось изображение размером 512 × 512"

plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray", vmin=0, vmax=255)
plt.title("Исходное изображение camera (512 × 512)")
plt.axis("off")
plt.show()

## 3. Теоретическая основа преобразования Хаара

Двумерное преобразование Хаара обрабатывает изображение по парам пикселей по горизонтали и вертикали. В результате на каждом уровне получаются четыре поддиапазона:

- **LL** — низкочастотная составляющая, содержащая основную информацию об изображении;
- **LH, HL, HH** — высокочастотные составляющие, описывающие детали и изменения яркости в разных направлениях.

Для блока пикселей

\[
\begin{pmatrix}a & b\\c & d\end{pmatrix}
\]

один из распространённых вариантов нормировки коэффициентов Хаара имеет вид:

\[
LL=\frac{a+b+c+d}{2},\quad
LH=\frac{a-b+c-d}{2},
\]

\[
HL=\frac{a+b-c-d}{2},\quad
HH=\frac{a-b-c+d}{2}.
\]

На следующем уровне преобразование применяется только к поддиапазону LL предыдущего уровня. Поэтому при исходном размере 512 × 512 размеры LL будут последовательно уменьшаться: 256 × 256, 128 × 128 и 64 × 64.

В `PyWavelets` функция `wavedec2` возвращает коэффициенты в структуре: `[cA3, (cH3, cV3, cD3), (cH2, cV2, cD2), (cH1, cV1, cD1)]`. Здесь `cA3` — низкочастотная составляющая третьего уровня.

## 4. Трёхуровневое преобразование Хаара

In [ ]:
# Выполняем двумерное дискретное вейвлет-преобразование Хаара
wavelet_name = "haar"
level = 3

coeffs = pywt.wavedec2(image.astype(np.float64),
                       wavelet=wavelet_name,
                       level=level,
                       mode="periodization")

# Низкочастотная составляющая третьего уровня
cA3 = coeffs[0]

print("Размер LL3:", cA3.shape)
for i, detail in enumerate(reversed(coeffs[1:]), start=1):
    print(f"Уровень {i}: размеры высокочастотных поддиапазонов:",
          [part.shape for part in detail])

## 5. Визуализация коэффициентов преобразования

In [ ]:
# Собираем все поддиапазоны в одну картину
coeffs_array, coeffs_slices = pywt.coeffs_to_array(coeffs)

plt.figure(figsize=(9, 9))
plt.imshow(coeffs_array, cmap="gray")
plt.title("Коэффициенты трёхуровневого преобразования Хаара")
plt.axis("off")
plt.colorbar(label="Значение коэффициента")
plt.show()

# Отдельно показываем LL после каждого уровня
ll1 = pywt.dwt2(image.astype(np.float64), "haar", mode="periodization")[0]
ll2 = pywt.dwt2(ll1, "haar", mode="periodization")[0]
ll3 = pywt.dwt2(ll2, "haar", mode="periodization")[0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, arr, title in zip(
    axes,
    [image, ll1, ll2, ll3],
    ["Исходное 512×512", "LL1 256×256", "LL2 128×128", "LL3 64×64"]
):
    ax.imshow(arr, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Пороговое сжатие коэффициентов

Для демонстрации сжатия оставим низкочастотную составляющую `LL3` без изменений, а среди всех высокочастотных коэффициентов сохраним 20% коэффициентов с наибольшими абсолютными значениями. Остальные коэффициенты заменим нулями.

Это **сжатие с потерями**: часть деталей изображения удаляется. Порог вычисляется по объединённому набору высокочастотных коэффициентов всех трёх уровней.

In [ ]:
# Копируем коэффициенты, чтобы не менять исходное разложение
compressed_coeffs = [coeffs[0].copy()]

# Объединяем все высокочастотные коэффициенты для расчёта порога
detail_values = np.concatenate([
    band.ravel()
    for level_details in coeffs[1:]
    for band in level_details
])

# 80-й процентиль означает, что примерно 20% модулей коэффициентов
# находятся выше порога (при совпадающих значениях доля может отличаться)
threshold = np.percentile(np.abs(detail_values), 80)
print(f"Пороговое значение: {threshold:.4f}")

# Обнуляем высокочастотные коэффициенты, модуль которых меньше порога
for level_details in coeffs[1:]:
    compressed_level = []
    for band in level_details:
        new_band = band.copy()
        new_band[np.abs(new_band) < threshold] = 0
        compressed_level.append(new_band)
    compressed_coeffs.append(tuple(compressed_level))

print("Пороговая обработка выполнена.")

## 7. Обратное преобразование и восстановление изображения

In [ ]:
restored_image_float = pywt.waverec2(
    compressed_coeffs,
    wavelet=wavelet_name,
    mode="periodization"
)

# На случай возможного отличия размеров после обратного преобразования
restored_image_float = restored_image_float[:image.shape[0], :image.shape[1]]

# Ограничиваем яркость допустимым диапазоном и преобразуем к uint8
restored_image = np.clip(np.rint(restored_image_float), 0, 255).astype(np.uint8)

print("Размер восстановленного изображения:", restored_image.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Исходное изображение")
axes[0].axis("off")

axes[1].imshow(restored_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("После сжатия и восстановления")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 8. Оценка результата

Для оценки качества восстановления используем PSNR (Peak Signal-to-Noise Ratio) и среднеквадратичную ошибку MSE. Чем выше PSNR и ниже MSE, тем ближе восстановленное изображение к исходному.

Также посчитаем долю ненулевых коэффициентов после пороговой обработки. Это показывает разреженность набора коэффициентов, но **не является точным размером сжатого файла**: для реального уменьшения файла нужно дополнительно закодировать коэффициенты.

In [ ]:
# Собираем коэффициенты до и после пороговой обработки
original_array, _ = pywt.coeffs_to_array(coeffs)
compressed_array, _ = pywt.coeffs_to_array(compressed_coeffs)

total_coefficients = compressed_array.size
nonzero_coefficients = np.count_nonzero(compressed_array)
retained_percent = 100 * nonzero_coefficients / total_coefficients
zeroed_percent = 100 - retained_percent

mse_value = mean_squared_error(image, restored_image)
psnr_value = peak_signal_noise_ratio(image, restored_image, data_range=255)

print("Общее количество коэффициентов:", total_coefficients)
print("Количество ненулевых коэффициентов:", nonzero_coefficients)
print(f"Ненулевых коэффициентов: {retained_percent:.2f}%")
print(f"Обнулено коэффициентов: {zeroed_percent:.2f}%")
print(f"MSE: {mse_value:.4f}")
print(f"PSNR: {psnr_value:.2f} dB")

## 9. Итоговая визуализация

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Исходное изображение")
axes[0].axis("off")

# Для наглядности используем логарифмический масштаб модулей коэффициентов
axes[1].imshow(np.log1p(np.abs(coeffs_array)), cmap="gray")
axes[1].set_title("Коэффициенты Хаара (log)")
axes[1].axis("off")

axes[2].imshow(restored_image, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Восстановленное изображение")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 10. Вывод

В ходе работы было выполнено трёхуровневое двумерное преобразование Хаара для стандартного изображения `camera` размером 512 × 512 пикселей из библиотеки `scikit-image`.

На каждом уровне изображение разделялось на низкочастотную и три высокочастотные составляющие. На последующих уровнях преобразование применялось к низкочастотной составляющей предыдущего уровня. В результате размер LL уменьшился с 512 × 512 до 64 × 64 пикселей.

Для демонстрации сжатия была выполнена пороговая обработка высокочастотных коэффициентов: коэффициенты с малыми абсолютными значениями были заменены нулями, а низкочастотная составляющая третьего уровня оставлена без изменений. Затем было выполнено обратное преобразование Хаара.

Качество восстановленного изображения оценено с помощью MSE и PSNR. Эксперимент показывает, что преобразование Хаара позволяет представить изображение в виде частотных составляющих и уменьшить количество ненулевых коэффициентов, сохраняя основные визуальные особенности изображения.